In [ ]:
!pip install -q lightgbm xgboost catboost optuna scikit-learn

import os
import gc
import random
import numpy as np
import pandas as pd
from scipy.optimize import minimize

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import torch

import warnings
warnings.filterwarnings('ignore')

SEED = 42
N_SPLITS = 5
TARGET_COL = 'addicted_label'
ID_COL = 'id'

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

In [ ]:
train_candidates = ['train.csv', 'data/raw/train.csv', 'predicting_smartphone_addiction/data/raw/train.csv', '../data/raw/train.csv']
test_candidates = ['test.csv', 'data/raw/test.csv', 'predicting_smartphone_addiction/data/raw/test.csv', '../data/raw/test.csv']

train_path = next((p for p in train_candidates if os.path.exists(p)), None)
test_path = next((p for p in test_candidates if os.path.exists(p)), None)

if train_path is None or test_path is None:
    from google.colab import files
    uploaded = files.upload()
    train_path, test_path = 'train.csv', 'test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
print(train_df.shape, test_df.shape)

In [ ]:
def engineer_features(df: pd.DataFrame, is_train: bool = True, cat_freq_maps: dict = None):
    out = df.copy()
    raw_features = [c for c in out.columns if c not in [ID_COL, TARGET_COL]]
    
    out['null_count'] = out[raw_features].isnull().sum(axis=1).astype(np.float32)
    for c in raw_features:
        if out[c].isnull().any():
            out[f'{c}_isna'] = out[c].isnull().astype(np.float32)
            
    social = out['social_media_hours']
    gaming = out['gaming_hours']
    work = out['work_study_hours']
    daily = out['daily_screen_time_hours']
    weekend = out['weekend_screen_time']
    sleep = out['sleep_hours']
    opens = out['app_opens_per_day']
    notifs = out['notifications_per_day']
    
    out['breakdown_sum'] = social.fillna(0) + gaming.fillna(0) + work.fillna(0)
    out['unaccounted_screen_time'] = daily - out['breakdown_sum']
    out['breakdown_to_screen_ratio'] = out['breakdown_sum'] / (daily + 1e-5)
    out['social_to_screen_ratio'] = social / (daily + 1e-5)
    out['gaming_to_screen_ratio'] = gaming / (daily + 1e-5)
    out['work_to_screen_ratio'] = work / (daily + 1e-5)
    out['entertainment_hours'] = social.fillna(0) + gaming.fillna(0)
    out['entertainment_to_work_ratio'] = out['entertainment_hours'] / (work.fillna(0) + 1e-5)
    
    out['weekend_to_daily_ratio'] = weekend / (daily + 1e-5)
    out['weekend_daily_diff'] = weekend - daily
    
    out['waking_hours'] = 24.0 - sleep
    out['screen_fraction_of_waking_day'] = daily / (out['waking_hours'] + 1e-5)
    out['non_screen_waking_hours'] = out['waking_hours'] - daily
    
    out['time_per_open'] = (daily * 60.0) / (opens + 1e-5)
    out['notifications_per_open'] = notifs / (opens + 1e-5)
    out['notifications_per_screen_hour'] = notifs / (daily + 1e-5)
    out['opens_per_screen_hour'] = opens / (daily + 1e-5)
    
    stress_map = {'Low': 0.0, 'Medium': 1.0, 'High': 2.0}
    impact_map = {'No': 0.0, 'Yes': 1.0}
    stress_num = out['stress_level'].map(stress_map)
    impact_num = out['academic_work_impact'].map(impact_map)
    out['stress_numeric'] = stress_num
    out['academic_impact_numeric'] = impact_num
    out['stress_x_screen'] = stress_num * daily
    out['impact_x_screen'] = impact_num * daily
    
    cat_cols = ['gender', 'stress_level', 'academic_work_impact']
    if is_train:
        cat_freq_maps = {}
        for c in cat_cols:
            freq = out[c].value_counts(normalize=True, dropna=True).to_dict()
            cat_freq_maps[c] = freq
            out[f'{c}_freq'] = out[c].map(freq).fillna(0.0).astype(np.float32)
            out[c] = out[c].astype('category')
        return out, cat_freq_maps
    else:
        for c in cat_cols:
            freq = cat_freq_maps.get(c, {})
            out[f'{c}_freq'] = out[c].map(freq).fillna(0.0).astype(np.float32)
            out[c] = out[c].astype('category')
        return out

train_feat, freq_maps = engineer_features(train_df, is_train=True)
test_feat = engineer_features(test_df, is_train=False, cat_freq_maps=freq_maps)

feature_cols = [c for c in train_feat.columns if c not in [ID_COL, TARGET_COL]]
cat_cols = [c for c in feature_cols if train_feat[c].dtype.name in ['category', 'object']]

In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
X = train_feat[feature_cols].copy()
y = train_feat[TARGET_COL].to_numpy()
X_test = test_feat[feature_cols].copy()

oof_predictions = {}
test_predictions = {}

lgb_oof = np.zeros(len(X))
lgb_test = np.zeros(len(X_test))
lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'n_estimators': 2000,
    'learning_rate': 0.03,
    'num_leaves': 63,
    'max_depth': 7,
    'subsample': 0.8,
    'colsample_bytree': 0.75,
    'random_state': SEED,
    'verbose': -1,
    'n_jobs': -1
}
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y[train_idx]
    X_val, y_val = X.iloc[val_idx], y[val_idx]
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50, verbose=False)])
    val_preds = model.predict_proba(X_val)[:, 1]
    lgb_oof[val_idx] = val_preds
    lgb_test += model.predict_proba(X_test)[:, 1] / N_SPLITS

oof_predictions['LightGBM'] = lgb_oof
test_predictions['LightGBM'] = lgb_test
print('LGBM AUC:', roc_auc_score(y, lgb_oof))

In [ ]:
xgb_oof = np.zeros(len(X))
xgb_test = np.zeros(len(X_test))
xgb_params = {
    'n_estimators': 2000,
    'learning_rate': 0.03,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.75,
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'enable_categorical': True,
    'random_state': SEED,
    'n_jobs': -1
}
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y[train_idx]
    X_val, y_val = X.iloc[val_idx], y[val_idx]
    model = xgb.XGBClassifier(**xgb_params, early_stopping_rounds=50)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    val_preds = model.predict_proba(X_val)[:, 1]
    xgb_oof[val_idx] = val_preds
    xgb_test += model.predict_proba(X_test)[:, 1] / N_SPLITS

oof_predictions['XGBoost'] = xgb_oof
test_predictions['XGBoost'] = xgb_test
print('XGB AUC:', roc_auc_score(y, xgb_oof))

In [ ]:
cb_oof = np.zeros(len(X))
cb_test = np.zeros(len(X_test))
X_cb = X.copy()
X_test_cb = X_test.copy()
for c in cat_cols:
    X_cb[c] = X_cb[c].astype(str).fillna('missing')
    X_test_cb[c] = X_test_cb[c].astype(str).fillna('missing')

cb_params = {
    'iterations': 1500,
    'learning_rate': 0.04,
    'depth': 6,
    'eval_metric': 'AUC',
    'random_seed': SEED,
    'verbose': 0,
    'task_type': 'GPU' if torch.cuda.is_available() else 'CPU'
}
for fold, (train_idx, val_idx) in enumerate(skf.split(X_cb, y)):
    X_train, y_train = X_cb.iloc[train_idx], y[train_idx]
    X_val, y_val = X_cb.iloc[val_idx], y[val_idx]
    model = cb.CatBoostClassifier(**cb_params, early_stopping_rounds=50)
    model.fit(X_train, y_train, eval_set=(X_val, y_val), cat_features=cat_cols, verbose=False)
    val_preds = model.predict_proba(X_val)[:, 1]
    cb_oof[val_idx] = val_preds
    cb_test += model.predict_proba(X_test_cb)[:, 1] / N_SPLITS

oof_predictions['CatBoost'] = cb_oof
test_predictions['CatBoost'] = cb_test
print('CB AUC:', roc_auc_score(y, cb_oof))

In [ ]:
model_names = list(oof_predictions.keys())
oof_matrix = np.column_stack([oof_predictions[m] for m in model_names])
test_matrix = np.column_stack([test_predictions[m] for m in model_names])

def loss_func(weights):
    weights = np.array(weights)
    weights = weights / np.sum(weights)
    blend = np.dot(oof_matrix, weights)
    return -roc_auc_score(y, blend)

init_weights = [1.0 / len(model_names)] * len(model_names)
bounds = [(0.0, 1.0) for _ in range(len(model_names))]
constraints = ({'type': 'eq', 'fun': lambda w: 1.0 - sum(w)})
res = minimize(loss_func, init_weights, method='SLSQP', bounds=bounds, constraints=constraints)
best_weights = res.x / np.sum(res.x)

ensemble_oof = np.dot(oof_matrix, best_weights)
ensemble_auc = roc_auc_score(y, ensemble_oof)
print('Weights:', dict(zip(model_names, best_weights.round(4))))
print('Ensemble AUC:', ensemble_auc)

final_test_preds = np.dot(test_matrix, best_weights)
submission_df = pd.DataFrame({
    ID_COL: test_df[ID_COL],
    TARGET_COL: np.clip(final_test_preds, 0.0, 1.0)
})
submission_df.to_csv('submission.csv', index=False)
print('Saved submission.csv', submission_df.shape)